# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why
**Method: Random Forest Regressor**, compared against a simpler Linear Regression as a sanity
check, and against the Week-4 baseline (position-tier average CTR).

**Why Random Forest:** the CTR/Engagement Opportunity Scoring task (from ML-03) is regression —
predicting a continuous CTR value from position plus content features. Random Forest handles
non-linear relationships and feature interactions (e.g. word count may only matter at certain
positions) without needing manual interaction terms, and it gives interpretable feature
importances afterward — matching this week's menu and next week's permutation-importance work.

**Why also fit Linear Regression:** per "does not reward complexity alone" — if a simple linear
model gets nearly the same score as the forest, that's an important, honest finding, not a
failure. The comparison table below reports both.

**What "beating the baseline" means here:** the Week-4 baseline predicts CTR using only a
lookup table of position-tier averages. Any model here must beat that same baseline, computed
fold-by-fold on the same grouped split (not reused from the ML-07 run), for the comparison to
be fair rather than circular.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

data = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_clicks)::DOUBLE / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_observed,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.gsc_impressions) AS impressions_total,
        ANY_VALUE(d.word_count) AS word_count,
        ANY_VALUE(d.content_type) AS content_type,
        ANY_VALUE(d.search_volume) AS search_volume,
        ANY_VALUE(d.competition) AS competition,
        DATE '2026-03-31' - ANY_VALUE(d.content_created_date) AS age_days
    FROM {FACT} f
    JOIN {DIM_CONTENT} d
      ON f.content_hash_id = d.content_hash_id AND f.client_hash_id = d.client_hash_id
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
      AND d.is_published IS TRUE
      AND d.is_deleted IS NOT TRUE
    GROUP BY f.content_hash_id, f.client_hash_id
""").df().dropna(subset=['ctr_observed', 'avg_position', 'word_count', 'age_days'])

def position_tier(p):
    if p <= 3: return '1_top_3'
    elif p <= 10: return '2_page_1'
    elif p <= 20: return '3_striking'
    elif p <= 50: return '4_page_3_5'
    else: return '5_deep'
data['position_tier'] = data['avg_position'].apply(position_tier)

print(f"Modeling frame: {len(data)} rows, {data['client_hash_id'].nunique()} clients")
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling frame: 121394 rows, 46 clients


,content_hash_id,client_hash_id,ctr_observed,avg_position,impressions_total,word_count,content_type,search_volume,competition,age_days,position_tier
0,content_14b1a02c1b8557fb,client_2094c6eb080311d5,0.000000,28.426940,90.0,4279,keyword article,0,0.00,110,4_page_3_5
1,content_15565677b6e1792f,client_2094c6eb080311d5,0.006757,12.350803,148.0,3011,keyword article,30,0.87,20,3_striking
2,content_15770c63daac443b,client_2094c6eb080311d5,0.001547,6.251382,2585.0,2694,keyword article,0,0.00,55,2_page_1
3,content_157db5a38382c639,client_2094c6eb080311d5,0.003500,6.104022,1143.0,3018,keyword article,0,0.00,55,2_page_1
4,content_15bc2acb5666f4ab,client_2094c6eb080311d5,0.000000,17.763790,53.0,2762,keyword article,30,0.83,20,3_striking


## 2. Split design

**Split: GroupKFold by `client_hash_id`, 5 folds.**

Content items from the same client likely share hidden site-level characteristics (domain
authority, brand recognition, template quality) that a random split would let leak across
train/test — the model could partly "memorize" a client's baseline CTR level rather than
learning genuine feature-CTR relationships. Grouping by client means every fold tests on
clients the model has never seen during that fold's training, which is the honest question:
does this generalize to a new client, not just a new page from a familiar one.

This mirrors the `client_holdout` split strategy used in the Week-1 starter pipeline
(`scripts/03_train_model.py`), applied here to the real warehouse data.

In [2]:
from sklearn.model_selection import GroupKFold

X_cols_numeric = ['avg_position', 'word_count', 'search_volume', 'competition', 'age_days']
X_cols_categorical = ['content_type']
y_col = 'ctr_observed'

groups = data['client_hash_id']
gkf = GroupKFold(n_splits=5)

fold_sizes = []
for i, (train_idx, test_idx) in enumerate(gkf.split(data, data[y_col], groups=groups)):
    train_clients = set(data.iloc[train_idx]['client_hash_id'])
    test_clients = set(data.iloc[test_idx]['client_hash_id'])
    overlap = train_clients & test_clients
    fold_sizes.append((len(train_idx), len(test_idx), len(overlap)))
    print(f"Fold {i+1}: train={len(train_idx)} rows, test={len(test_idx)} rows, client overlap={len(overlap)} (should be 0)")


Fold 1: train=97119 rows, test=24275 rows, client overlap=0 (should be 0)
Fold 2: train=97105 rows, test=24289 rows, client overlap=0 (should be 0)
Fold 3: train=97117 rows, test=24277 rows, client overlap=0 (should be 0)
Fold 4: train=97117 rows, test=24277 rows, client overlap=0 (should be 0)
Fold 5: train=97118 rows, test=24276 rows, client overlap=0 (should be 0)


## 3. Train + compare vs my baseline

**Comparison table below:** baseline (position-tier lookup) vs Linear Regression vs Random
Forest, all evaluated out-of-fold on the identical GroupKFold-by-client split, same metric
(MAE), so the comparison is fair — no model is scored on data its own fold saw during training,
and the baseline is recomputed per-fold too (not reused from ML-07's single global fit).
**Results:**

| Method | MAE | R² | Spearman | MAE improvement vs baseline |
|---|---|---|---|---|
| Baseline (position-tier) | 0.008499 | -0.010 | **0.166** | — |
| Linear Regression | 0.008002 | -0.008 | 0.073 | +5.8% |
| Random Forest | 0.007749 | -0.022 | 0.057 | **+8.8%** |

**This result is genuinely mixed, not a clean win — and that's worth reporting honestly rather
than only citing the flattering number.**

On raw MAE, both models beat the baseline, Random Forest most of all (8.8% lower error).
But two things temper that:

1. **All three R² values are negative.** A negative R² means the model explains *less* variance
   than simply predicting the average CTR for every row would. None of these features — even
   in combination, even with a Random Forest — capture much of what drives CTR in this data.
   This matches the very weak `avg_position`-`ctr` correlation (-0.049) already found in ML-04.

2. **On Spearman rank correlation, the baseline actually wins** (0.166 vs 0.073 and 0.057).
   This matters more than it might look, because this lane's actual deliverable is a *ranked*
   action queue (ML-07) — what matters operationally is getting the ranking order right, not
   minimizing average absolute error. By that measure, the simple position-tier lookup ranks
   pages more sensibly than either learned model does.

**Honest conclusion:** Random Forest wins on MAE but loses on the ranking metric this lane
actually depends on. Given "does not reward complexity alone," the fair takeaway is that
neither model is a clear upgrade over the baseline for this lane's real purpose — CTR appears
to be substantially noisier than these features can capture, at least at this feature set and
this monthly grain.

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr

X_cols_numeric = ['avg_position', 'word_count', 'search_volume', 'competition', 'age_days']
X_cols_categorical = ['content_type']
y_col = 'ctr_observed'

data['search_volume'] = data['search_volume'].astype('float64').fillna(0)
data['competition'] = data['competition'].astype('float64').fillna(0)
data['word_count'] = data['word_count'].astype('float64').fillna(0)
data['age_days'] = data['age_days'].astype('float64').fillna(0)

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), X_cols_categorical),
], remainder='passthrough')

models = {
    'linear_regression': Pipeline([('prep', preprocess), ('model', LinearRegression())]),
    'random_forest': Pipeline([('prep', preprocess), ('model', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))]),
}

X = data[X_cols_numeric + X_cols_categorical]
y = data[y_col].values
groups = data['client_hash_id']
gkf = GroupKFold(n_splits=5)

results = {name: np.zeros(len(data)) for name in models}
baseline_oof = np.zeros(len(data))

for train_idx, test_idx in gkf.split(X, y, groups=groups):
    tier_lookup = data.iloc[train_idx].groupby('position_tier')[y_col].mean()
    baseline_oof[test_idx] = data.iloc[test_idx]['position_tier'].map(tier_lookup).fillna(tier_lookup.mean())

    for name, pipe in models.items():
        pipe.fit(X.iloc[train_idx], y[train_idx])
        results[name][test_idx] = pipe.predict(X.iloc[test_idx])

rows = []
rows.append(['baseline_position_tier', mean_absolute_error(y, baseline_oof), r2_score(y, baseline_oof), spearmanr(y, baseline_oof).correlation])
for name, preds in results.items():
    rows.append([name, mean_absolute_error(y, preds), r2_score(y, preds), spearmanr(y, preds).correlation])

comparison = pd.DataFrame(rows, columns=['method', 'MAE', 'R2', 'spearman'])
comparison['MAE_improvement_vs_baseline_pct'] = (
    (comparison.loc[0, 'MAE'] - comparison['MAE']) / comparison.loc[0, 'MAE'] * 100
)
print(comparison.to_string(index=False))

                method      MAE        R2  spearman  MAE_improvement_vs_baseline_pct
baseline_position_tier 0.008499 -0.010338  0.166072                         0.000000
     linear_regression 0.008002 -0.007607  0.072733                         5.846060
         random_forest 0.007749 -0.022438  0.057163                         8.820747


## 4. Errors and interpretation
**What the model leans on, and where it's most wrong** — since neither model clearly beat the
baseline on ranking, this section digs into *why*, rather than just reporting more numbers.
**Feature importance:** `word_count` dominates (51%), followed by `avg_position` and `age_days`
(roughly 22% each) — together these three account for ~95% of what the model leans on.
`content_type`, `competition`, and `search_volume` contribute very little.

**Error pattern by position tier:** errors are largest at `1_top_3` (MAE 0.021) and shrink
steadily down through `5_deep` (MAE 0.004) — the model struggles most exactly where CTR is
most volatile (small changes in top-ranked clicks swing CTR a lot), and is most stable where
CTR is naturally low and flat.

**The three largest individual errors, investigated rather than just reported:** all three
(and in fact all of the top 10 worst-predicted rows) share one thing — `impressions_total = 1`,
meaning a single impression that happened to convert to a single click, producing a "perfect"
100% CTR. Confirmed directly: rows with ≤3 impressions have 4.5x the average error of
higher-volume rows (0.024 vs 0.005). This is not a modeling failure — a model that predicted
100% CTR for a 1-impression page would be overfitting to noise, not learning a real pattern.
The honest takeaway is that this feature set is unreliable at the lowest end of the impression
range, and a future iteration should consider an impression floor before including a row in
training at all, rather than asking the model to explain single-click outcomes.

**Tying back to Section 3's ranking result:** combined with the baseline's stronger Spearman
correlation, the picture is consistent — this feature set (position, word count, age, content
type) captures some real signal (word_count and position both show up as meaningfully important,
not noise), but not enough to reliably outrank a simple position-tier lookup for this lane's
actual ranking purpose. More/better features (e.g. title-length proxies, real query-intent
signals) would likely matter more than trying more complex model architectures on this same
feature set.

In [6]:
# Feature importance from the Random Forest (fit once on all data, for interpretation only)
rf_full = Pipeline([('prep', preprocess), ('model', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))])
rf_full.fit(X, y)

feature_names = rf_full.named_steps['prep'].get_feature_names_out()
importances = rf_full.named_steps['model'].feature_importances_
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
print("Feature importances (Random Forest):")
print(imp_df.to_string(index=False))

# Where are the Random Forest's out-of-fold errors biggest? By position tier.
data['rf_pred'] = results['random_forest']
data['abs_error'] = (data['ctr_observed'] - data['rf_pred']).abs()
error_by_tier = data.groupby('position_tier')['abs_error'].agg(['mean', 'size']).rename(columns={'size': 'n'})
print("\nMean absolute error by position tier (Random Forest, out-of-fold):")
print(error_by_tier)

# Three concrete worst predictions
worst = data.nlargest(3, 'abs_error')[['content_hash_id', 'position_tier', 'ctr_observed', 'rf_pred', 'impressions_total', 'word_count']]
print("\nThree largest individual errors:")
print(worst.to_string(index=False))
# Check: are top_3's biggest errors driven by very low impression counts (unreliable CTR)?
low_impression_check = data.nlargest(10, 'abs_error')[['position_tier', 'ctr_observed', 'rf_pred', 'impressions_total']]
print("Top 10 worst errors, with impression counts:")
print(low_impression_check.to_string(index=False))

print(f"\nRows with impressions_total <= 3: {(data['impressions_total'] <= 3).sum()} of {len(data)}")
print(f"Mean abs_error for impressions_total <= 3: {data[data['impressions_total'] <= 3]['abs_error'].mean():.4f}")
print(f"Mean abs_error for impressions_total > 3: {data[data['impressions_total'] > 3]['abs_error'].mean():.4f}")


Feature importances (Random Forest):
                             feature  importance
               remainder__word_count    0.510235
             remainder__avg_position    0.218594
                 remainder__age_days    0.218254
              remainder__competition    0.016160
            remainder__search_volume    0.013507
    cat__content_type_feedly article    0.012150
   cat__content_type_keyword article    0.010586
cat__content_type_comparison article    0.000514

Mean absolute error by position tier (Random Forest, out-of-fold):
                   mean      n
position_tier                 
1_top_3        0.020981  12651
2_page_1       0.007312  62900
3_striking     0.005046  22717
4_page_3_5     0.004441  19881
5_deep         0.003827   3245

Three largest individual errors:
         content_hash_id position_tier  ctr_observed  rf_pred  impressions_total  word_count
content_5b12e81bfdba5de5        5_deep           1.0 0.000579                1.0       828.0
content_b11dddc3f

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.